# Thí nghiệm 5-B — chạy lại ba biến thể bằng **đúng phiếu từng chữ**

Lượt A (17/08) đi lệch phiếu ở ba chỗ, tất cả đều do bám theo cấu hình TN3 để giữ khả
năng tái lập 12,71%. Notebook này chạy **đúng như phiếu viết**, không bám TN3 nữa:

| # | Phiếu TN5 | Lượt A đã chạy | Lượt B (notebook này) |
|---|---|---|---|
| 1 | `PROMPT_V1` kết bằng *"Chỉ trả về JSON."* | *"…JSON **theo mẫu**."* (bản TN3) | **đúng phiếu §3.1** |
| 2 | `MAX_TOKENS['V1_baseline'] = 250` | 200 (bằng TN3) | **250** |
| 3 | `inputs = tokenizer(prompt, …)` thô | `apply_chat_template` + `TERMINATORS` | **thô, đúng §4.1** |

Chỗ 3 áp cho **cả ba biến thể**, nên phải chạy lại đủ ba chứ không riêng V1.

**Cảnh báo trước khi chạy.** Phiếu §5.3 B đòi V1 tái lập 12,71% của TN3, nhưng TN3 chạy
ở 200 token **và** qua `apply_chat_template`. Chạy đúng phiếu thì hai điều kiện ấy không
còn, nên **V1 nhiều khả năng KHÔNG còn khớp 12,71%**. Đó là hệ quả của mâu thuẫn nội tại
trong phiếu, không phải lỗi cấu hình. Phần 12 đối chiếu A ↔ B để thầy thấy rõ mức lệch.

**Runtime:** T4. V1 ~26 phút, V2 ~16 phút, V3 ~75 phút → tổng khoảng 2 giờ.
Lưu CSV sau mỗi biến thể; Colab ngắt giữa chừng chỉ mất biến thể đang chạy.

## 1. Kiểm GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), \
    "KHONG CO GPU. Vao Runtime -> Change runtime type -> T4 GPU roi chay lai."
p = torch.cuda.get_device_properties(0)
print(f"\nGPU : {p.name}")
print(f"VRAM: {p.total_memory/1e9:.2f} GB")

## 2. Cài thư viện — giữ nguyên như lượt A (Llama-3 chạy tốt trên bản mới)

In [ ]:
%pip install -q -U "transformers>=4.43" accelerate bitsandbytes huggingface_hub python-docx

import transformers, torch
print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)

## 3. Nạp dữ liệu

Upload **`tn5b_input.zip`** — 10 tệp của gói TN5 cộng ba tệp per-query của lượt A
(`tn5a_perquery_*.csv`) để Phần 12 đối chiếu A ↔ B.

In [ ]:
import os, zipfile
from google.colab import files

os.makedirs('/content/tn5b_data', exist_ok=True)
up = files.upload()
for ten in up:
    if ten.endswith('.zip'):
        with zipfile.ZipFile(ten) as z:
            z.extractall('/content/tn5b_data')
    else:
        os.replace(ten, f'/content/tn5b_data/{ten}')
print()
for f in sorted(os.listdir('/content/tn5b_data')):
    print(' ', f)

## 4. Đọc và kiểm dữ liệu — 181 / 13.081 / 118–63

In [ ]:
MODEL_ID = 'NousResearch/Meta-Llama-3-8B-Instruct'   # y nguyen TN3/luot A

import os
import pandas as pd

DATA = '/content/tn5b_data'
OUT  = '/content/tn5b_ket_qua'
os.makedirs(OUT, exist_ok=True)

df_test = pd.read_csv(f'{DATA}/independent_scored_perquery.csv')
df_icd  = pd.read_csv(f'{DATA}/ICD10_cleaned.csv')

valid_codes_full  = set(df_icd['Mã ICD'].astype(str).str.strip().str.upper())
valid_codes_3char = set(c[:3] for c in valid_codes_full)

n_reach = int((df_test['voi_toi_duoc'] == True).sum())
n_outr  = int((df_test['voi_toi_duoc'] == False).sum())
print(f"So ca test          : {len(df_test):>6}   (ky vong 181)")
print(f"So ma ICD-10 day du : {len(valid_codes_full):>6}   (ky vong 13081)")
print(f"Reachable / out     : {n_reach} / {n_outr}   (ky vong 118 / 63)")
assert len(df_test) == 181,            "SAI so ca test"
assert len(valid_codes_full) == 13081, "SAI so ma ICD"
assert (n_reach, n_outr) == (118, 63), "SAI ty le reachable"

CO_LUOT_A = all(os.path.exists(f'{DATA}/tn5a_perquery_{v}.csv')
                for v in ['V1_baseline', 'V2_english', 'V3_cot'])
print("\nCo ket qua luot A de doi chieu:", "CO" if CO_LUOT_A else "KHONG (Phan 12 rut gon)")

## 5. Ba prompt — **nguyên văn phiếu §3.1–3.3**

`PROMPT_V1` ở đây khác lượt A đúng một cụm: quy tắc 4 là **`Chỉ trả về JSON.`**
(lượt A dùng bản TN3: `Chỉ trả về JSON theo mẫu.`).
`PROMPT_V2` và `PROMPT_V3` giống lượt A vì lượt A vốn đã khớp phiếu từng chữ.

Ngoặc nhọn trong ví dụ JSON vẫn phải viết đôi `{{…}}` — phiếu in ngoặc đơn nên
`str.format()` sẽ ném `KeyError`. Đây là lỗi in ấn của phiếu, không phải khác biệt nội dung.

In [ ]:
PROMPT_V1 = """Bạn là bác sĩ trợ lý chuyên chẩn đoán bằng mã ICD-10.

NHIỆM VỤ: Đọc mô tả triệu chứng bệnh nhân, đưa ra danh sách TOP-10 mã ICD-10
có khả năng nhất, sắp xếp theo xác suất giảm dần.

QUY TẮC:
1. Mỗi mã ICD-10 phải là mã HỢP LỆ (định dạng: 1 chữ cái + 2-3 chữ số +
   tùy chọn "." + 1-2 chữ số. Ví dụ: A00, B07.9, K21.9).
2. Được phép dùng BẤT KỲ mã nào trong toàn bộ catalogue ICD-10 của
   Bộ Y tế Việt Nam (Quyết định 4469/QĐ-BYT 2020), gồm khoảng 13.081 mã.
3. Đưa mã 3 ký tự (phân nhóm) nếu không đủ tự tin về ký tự thứ 4.
4. KHÔNG giải thích. Chỉ trả về JSON.

VÍ DỤ:
Mô tả: "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ, đã 3 ngày."
Kết quả JSON: {{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}

---
Mô tả: "{query_text}"
Kết quả JSON:"""

In [ ]:
PROMPT_V2 = """You are a medical AI assistant specializing in ICD-10 coding.

TASK: Given a patient description (which may be in Vietnamese), output the TOP-10
most likely ICD-10 codes from the full Ministry of Health catalogue
(approximately 13,081 codes), sorted by probability descending.

RULES:
1. Each code must be a valid ICD-10 code (format: 1 letter + 2-3 digits
   + optional "." + 1-2 digits, e.g., A00, B07.9, K21.9).
2. Any code from the full ICD-10 catalogue is allowed. Do NOT restrict
   to any subset.
3. Return 3-character codes if unsure about the 4th character.
4. Output JSON only. No explanation.

EXAMPLE:
Description (Vietnamese): "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ."
Output: {{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}

---
Description: "{query_text}"
Output:"""

In [ ]:
PROMPT_V3 = """Bạn là bác sĩ trợ lý chuyên chẩn đoán ICD-10.

Đọc mô tả bệnh nhân và thực hiện suy luận từng bước:

Bước 1: Liệt kê 3-5 triệu chứng chính từ mô tả.
Bước 2: Nêu 2-3 chẩn đoán phân biệt có thể (differential diagnosis).
Bước 3: Cho mỗi chẩn đoán, nêu mã ICD-10 tương ứng.
Bước 4: Mở rộng thành TOP-10 mã ICD-10 sắp xếp theo xác suất giảm dần.

Cuối cùng, TRẢ VỀ JSON: {{"top10_icd": [...]}}

VÍ DỤ:
Mô tả: "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ, đã 3 ngày."
Bước 1: triệu chứng chính: đau bụng, buồn nôn, sốt nhẹ, kéo dài 3 ngày
Bước 2: viêm ruột thừa (K35), viêm dạ dày ruột (K52), nhiễm khuẩn tiêu hóa (A09)
Bước 3: K35.9, K52.9, A09
Bước 4: Top-10 sắp xếp theo probability:
{{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}

---
Mô tả: "{query_text}"
"""

### `MAX_TOKENS` — **đúng phiếu §4.1**, V1 là 250 chứ không phải 200

In [ ]:
PROMPTS = {'V1_baseline': PROMPT_V1, 'V2_english': PROMPT_V2, 'V3_cot': PROMPT_V3}
MAX_TOKENS = {'V1_baseline': 250, 'V2_english': 250, 'V3_cot': 600}   # dung phieu §4.1

import re, hashlib
for ten, tpl in PROMPTS.items():
    _ = tpl.format(query_text='<cau hoi>')
    print(f"  {ten:<12} format() OK | {len(tpl):>4} ky tu | max_new_tokens={MAX_TOKENS[ten]}")

# V1 phai KHAC ban TN3 dung o luot A dung mot cum "theo mau"
assert 'Chỉ trả về JSON.' in PROMPT_V1 and 'theo mẫu' not in PROMPT_V1, \
    "PROMPT_V1 khong dung ban phieu"
print("\nSHA-256 V1 (ban phieu):", hashlib.sha256(PROMPT_V1.encode()).hexdigest())
print("SHA-256 V1 (ban TN3, luot A): ca64ec593fe460cc78265902d84c2fdd4424d64683a333407f7d96f61c466722")

## 6. Bộ chấm — giữ y nguyên lượt A để hai lượt so được với nhau

In [ ]:
import json, re

def chuan_hoa_ma_icd(raw_code):
    if not raw_code:
        return None
    code = str(raw_code).strip().upper().rstrip('.')
    m = re.match(r'^([A-Z]\d{2}(?:\d)?(?:\.\d{1,2})?)$', code)
    return m.group(1) if m else None


def extract_json_safe(raw):
    m = re.search(r'\{[^{}]*"top10_icd"\s*:\s*\[[^\]]*\][^{}]*\}', raw)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {'top10_icd': re.findall(r'\b([A-Z]\d{2}(?:\.\d{1,2})?)\b', raw)[:10]}


def cham_1_ca(llm_output_raw, gold3, gold_full, valid_3char):
    raw_list = extract_json_safe(llm_output_raw).get('top10_icd', [])
    top_10   = [c for c in (chuan_hoa_ma_icd(x) for x in raw_list) if c][:10]
    top_10_3 = [c[:3] for c in top_10]
    g3 = str(gold3).strip().upper()
    gf = str(gold_full).strip().upper()
    try:
        rank_3char = top_10_3.index(g3) + 1
    except ValueError:
        rank_3char = 0
    return {
        'top_10'              : ' || '.join(top_10),
        'top_10_3char'        : ' || '.join(top_10_3),
        'hit1'                : bool(top_10_3 and top_10_3[0] == g3),
        'hit5'                : g3 in top_10_3[:5],
        'hit10'               : g3 in top_10_3[:10],
        'hit1_full_icd'       : bool(top_10 and top_10[0] == gf),
        'rank_3char'          : rank_3char,
        'n_ma_parse_duoc'     : len(top_10),
        'n_valid_in_catalogue': sum(1 for c in top_10_3 if c in valid_3char),
    }

_r = cham_1_ca('{"top10_icd": ["K21.9","R10","XX","J32.","A09"]}', 'R10', 'R10.4', valid_codes_3char)
assert _r['rank_3char'] == 2 and _r['n_ma_parse_duoc'] == 4, _r
print("Bo cham OK (giong luot A):", _r)

## 7. Nạp model — 4-bit NF4, y nguyên TN3 và lượt A

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, os

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = (userdata.get('HF_TOKEN') or '').strip() or None
except Exception:
    HF_TOKEN = (os.environ.get('HF_TOKEN') or '').strip() or None
print("Token HF:", "co" if HF_TOKEN else "khong (mirror NousResearch khong can)")

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map={"": 0}, token=HF_TOKEN)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Nap model xong!")

## 8. Hàm sinh — **nguyên văn phiếu §4.1**

Khác lượt A đúng một chỗ và là chỗ quan trọng nhất: **không** gọi `apply_chat_template`,
đưa thẳng prompt vào tokenizer. Cũng không khai `eos_token_id`, chỉ đặt `pad_token_id`,
đúng như phiếu viết.

Hệ quả biết trước: Llama-3-Instruct không nhận thẻ hội thoại nên chạy ở chế độ hoàn thành
văn bản; nó thường sinh dài hơn và hay viết thêm sau JSON. Đó chính là điều notebook này
đo — phiếu chỉ định như vậy thì kết quả ra sao.

In [ ]:
def sinh_predictions_variant(query_text, variant):
    """Nguyen van phieu §4.1: tokenizer tho, khong chat template, khong TERMINATORS."""
    prompt = PROMPTS[variant].format(query_text=query_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS[variant],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


_thu = sinh_predictions_variant(df_test.iloc[0]['query_text'], 'V1_baseline')
print("Thu V1 (250 token, tokenizer tho):")
print(_thu[:400])
assert _thu, "Model tra ve rong."

## 9. Dry run 5 ca × 3 biến thể

In [ ]:
import time

for variant in ['V1_baseline', 'V2_english', 'V3_cot']:
    print(f"\n{'='*70}\n--- {variant} (max_new_tokens={MAX_TOKENS[variant]}) ---")
    t0 = time.time()
    for _, row in df_test.head(5).iterrows():
        raw = sinh_predictions_variant(row['query_text'], variant)
        sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
        print(f"  {row['case_id']}: hit1={sc['hit1']} valid={sc['n_valid_in_catalogue']}/{sc['n_ma_parse_duoc']}")
        print(f"    top10 : {sc['top_10_3char']}")
        print(f"    raw   : {raw[:120].replace(chr(10), ' / ')}")
    print(f"  Toc do: {(time.time()-t0)/5:.1f} giay/ca -> uoc {(time.time()-t0)/5*181/60:.0f} phut")

## 10. Chạy full — 3 biến thể × 181 ca

In [ ]:
from tqdm.auto import tqdm
import hashlib

BO_QUA_NEU_CO_TEP = True
all_variant_results = {}

for variant in ['V1_baseline', 'V2_english', 'V3_cot']:
    fname = f'{OUT}/tn5b_perquery_llama3_{variant}.csv'
    if BO_QUA_NEU_CO_TEP and os.path.exists(fname):
        df_variant = pd.read_csv(fname); elapsed = float('nan')
        print(f"\n=== {variant}: da co tep, nap lai ===")
    else:
        print(f"\n=== RUNNING {variant} ===")
        t0, results = time.time(), []
        for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=variant):
            raw = sinh_predictions_variant(row['query_text'], variant)
            sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
            results.append({
                'case_id': row['case_id'], 'source_channel': row['source_channel'],
                'gold3': str(row['gold3']).strip().upper(),
                'gold_full': str(row['gold']).strip().upper(),
                'voi_toi_duoc': row['voi_toi_duoc'], 'variant': variant,
                'llm_raw': raw, **sc})
        df_variant = pd.DataFrame(results)
        df_variant.to_csv(fname, index=False, encoding='utf-8-sig')
        elapsed = (time.time() - t0) / 60

    with open(fname, 'rb') as f:
        sha = hashlib.sha256(f.read()).hexdigest()
    hit1 = int(df_variant['hit1'].astype(bool).sum())
    print(f"  Hit@1 : {hit1}/181 = {100*hit1/181:.2f}%")
    print(f"  0 ma  : {(df_variant['n_ma_parse_duoc'] == 0).sum()} ca")
    print(f"  SHA   : {sha}")
    all_variant_results[variant] = {'df': df_variant, 'sha': sha, 'time_min': elapsed}

## 11. Thống kê chéo — cùng công thức lượt A

In [ ]:
from scipy.stats import binomtest
from scipy import stats

def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return (0.0, 0.0)
    z = stats.norm.ppf(1 - alpha/2); p = k/n
    den = 1 + z**2/n
    c = (p + z**2/(2*n))/den
    h = z*((p*(1-p)/n + z**2/(4*n**2))**0.5)/den
    return (max(0, (c-h)*100), min(100, (c+h)*100))

med       = pd.read_csv(f'{DATA}/independent_scored_perquery.csv').set_index('case_id')
medkg_hit = (med['rank3'] == 1).astype(int)
mask_reach = (med['voi_toi_duoc'] == True).reindex(med.index).fillna(False)

def mcnemar(u_hit, o_hit, mask=None):
    u = u_hit.reindex(med.index).fillna(0).astype(int)
    o = o_hit.reindex(med.index).fillna(0).astype(int)
    if mask is not None:
        u, o = u[mask], o[mask]
    b = int(((u == 1) & (o == 0)).sum()); c = int(((u == 0) & (o == 1)).sum())
    p = binomtest(min(b, c), n=b+c, p=0.5).pvalue if (b+c) > 0 else 1.0
    return b, c, float(p)

variant_stats = []
for variant in ['V1_baseline', 'V2_english', 'V3_cot']:
    data = all_variant_results[variant]
    df_v = data['df'].set_index('case_id')
    v_hit = df_v['hit1'].astype(bool).astype(int)
    v_al  = v_hit.reindex(med.index).fillna(0).astype(int)
    b181, c181, p181 = mcnemar(v_hit, medkg_hit)
    b118, c118, p118 = mcnemar(v_hit, medkg_hit, mask=mask_reach)
    k = int(v_al.sum()); lo, hi = wilson_ci(k, 181)
    mrr = df_v['rank_3char'].apply(lambda x: 1/x if x > 0 else 0).mean()
    variant_stats.append({
        'variant': variant, 'max_new_tokens': MAX_TOKENS[variant],
        'hit1_pct_full': round(100*k/181, 2), 'hit1_ci': f'[{lo:.2f}, {hi:.2f}]',
        'hit1_pct_reach': round(100*v_al[mask_reach].sum()/118, 2),
        'hit1_pct_outreach': round(100*v_al[~mask_reach].sum()/63, 2),
        'hit5_pct': round(100*df_v['hit5'].astype(bool).sum()/181, 2),
        'mrr': round(float(mrr), 4),
        'n_parse_0_ma': int((df_v['n_ma_parse_duoc'] == 0).sum()),
        'avg_valid_top10': round(float(df_v['n_valid_in_catalogue'].mean()), 2),
        'mcnemar_n181_b': b181, 'mcnemar_n181_c': c181, 'mcnemar_n181_p': round(p181, 4),
        'mcnemar_n118_b': b118, 'mcnemar_n118_c': c118, 'mcnemar_n118_p': round(p118, 4),
        'holm_adj_m8': round(min(1.0, p118*8), 4),
        'sha256': data['sha'],
        'time_min': (round(data['time_min'], 1) if data['time_min'] == data['time_min'] else None),
    })

thu_tu = sorted(range(3), key=lambda i: variant_stats[i]['mcnemar_n118_p'])
truoc = 0.0
for hang, i in enumerate(thu_tu):
    adj = min(1.0, max(truoc, variant_stats[i]['mcnemar_n118_p'] * (3 - hang)))
    truoc = adj
    variant_stats[i]['holm_tuan_tu'] = round(adj, 4)

df_summary = pd.DataFrame(variant_stats)
df_summary.to_csv(f'{OUT}/tn5b_summary_variants.csv', index=False, encoding='utf-8-sig')
print(df_summary[['variant','hit1_pct_full','hit1_ci','hit1_pct_reach','hit1_pct_outreach',
                  'mrr','mcnemar_n118_b','mcnemar_n118_c','mcnemar_n118_p',
                  'holm_adj_m8','holm_tuan_tu','n_parse_0_ma']].to_string(index=False))

hit1s = [v['hit1_pct_full'] for v in variant_stats]
spread = max(hit1s) - min(hit1s)
KICH_BAN = ('A (<=2pp)' if spread <= 2 else 'B (2-5pp)' if spread <= 5 else 'C (>5pp)')
print(f"\nBien do Hit@1: {spread:.2f} pp -> kich ban {KICH_BAN}")

## 12. Đối chiếu lượt A ↔ lượt B — phần trả lời câu hỏi của notebook này

In [ ]:
print("=" * 84)
print(f"{'Bien the':<14}{'Hit@1 A':>10}{'Hit@1 B':>10}{'lech':>8}"
      f"{'p(118) A':>11}{'p(118) B':>11}{'0-ma A':>9}{'0-ma B':>9}")
print("=" * 84)

A_REF = {'V1_baseline': (12.71, 0.0041, 0), 'V2_english': (11.60, 0.0066, 0),
         'V3_cot': (9.39, 0.0768, 1)}   # so cua luot A, tu tn5_summary_variants.csv

s = df_summary.set_index('variant')
for v in ['V1_baseline', 'V2_english', 'V3_cot']:
    ha, pa, za = A_REF[v]
    hb = s.loc[v, 'hit1_pct_full']; pb = s.loc[v, 'mcnemar_n118_p']; zb = s.loc[v, 'n_parse_0_ma']
    print(f"{v:<14}{ha:>10.2f}{hb:>10.2f}{hb-ha:>+8.2f}{pa:>11.4f}{pb:>11.4f}{za:>9}{zb:>9}")

if CO_LUOT_A:
    print("\nMuc tung ca (so nguyen chuoi top-10):")
    for v in ['V1_baseline', 'V2_english', 'V3_cot']:
        a = pd.read_csv(f'{DATA}/tn5a_perquery_{v}.csv').set_index('case_id')['top_10_3char'].astype(str)
        b = all_variant_results[v]['df'].set_index('case_id')['top_10_3char'].astype(str)
        chung = a.index.intersection(b.index)
        print(f"  {v:<14}: {sum(1 for i in chung if a[i]==b[i])}/{len(chung)} ca giong het")

print("\n" + "=" * 84)
v1b = s.loc['V1_baseline', 'hit1_pct_full']
print(f"V1 chay dung phieu (250 token, tokenizer tho) = {v1b:.2f}%")
print(f"Moc TN3 ma phieu §5.3 B doi tai lap           = 12,71%")
print(f"Lech {abs(v1b-12.71):.2f} pp -> "
      + ("VAN TAI LAP DUOC." if abs(v1b-12.71) < 0.6 else
         "KHONG con tai lap. Day la he qua cua mau thuan trong phieu:\n"
         "  §4.1 doi 250 token va tokenizer tho, nhung TN3 chay 200 token qua chat template.\n"
         "  Hai yeu cau khong the cung dat. Xin thay chon mot."))
print("=" * 84)

## 13. Ghi chú diễn giải + băm SHA + đóng gói

In [ ]:
from docx import Document

doc = Document()
doc.add_heading('TN5-B — chạy lại ba biến thể đúng phiếu từng chữ', 0)
doc.add_paragraph(f"Model: {MODEL_ID} | 4-bit NF4 | greedy | n = 181 | "
                  f"tokenizer thô (không chat template), đúng phiếu §4.1")
doc.add_paragraph("Ba chỗ sửa so với lượt A: PROMPT_V1 đúng phiếu §3.1; "
                  "MAX_TOKENS['V1_baseline'] = 250; hàm sinh dùng tokenizer thô.")

doc.add_heading('A. Kết quả', level=1)
t = doc.add_table(rows=1, cols=6); t.style = 'Light Grid Accent 1'
for i, h in enumerate(['Biến thể', 'Hit@1 (%)', 'KTC 95%', 'Reachable (%)',
                       'Out-of-reach (%)', 'McNemar p (n=118)']):
    t.rows[0].cells[i].text = h
for v in ['V1_baseline', 'V2_english', 'V3_cot']:
    r = t.add_row().cells
    r[0].text = v
    r[1].text = str(s.loc[v, 'hit1_pct_full']); r[2].text = str(s.loc[v, 'hit1_ci'])
    r[3].text = str(s.loc[v, 'hit1_pct_reach']); r[4].text = str(s.loc[v, 'hit1_pct_outreach'])
    r[5].text = str(s.loc[v, 'mcnemar_n118_p'])
doc.add_paragraph(f"Biên độ Hit@1: {spread:.2f} pp — kịch bản {KICH_BAN}.")

doc.add_heading('B. Đối chiếu với lượt A', level=1)
for v in ['V1_baseline', 'V2_english', 'V3_cot']:
    ha, pa, _ = A_REF[v]
    doc.add_paragraph(f"{v}: lượt A {ha}% → lượt B {s.loc[v,'hit1_pct_full']}% "
                      f"(lệch {s.loc[v,'hit1_pct_full']-ha:+.2f} pp); "
                      f"p(n=118) {pa} → {s.loc[v,'mcnemar_n118_p']}.")
doc.add_paragraph(f"V1 so với mốc TN3 12,71%: lệch {abs(v1b-12.71):.2f} pp.")

doc.add_heading('C. Nhận định (người viết bổ sung)', level=1)
doc.add_paragraph('[…]')
doc.save(f'{OUT}/tn5b_ghichu_diengiai.docx')

dong = []
for f in sorted(os.listdir(OUT)):
    if f.endswith('SHA256.txt'):
        continue
    with open(f'{OUT}/{f}', 'rb') as fh:
        dong.append(f'{hashlib.sha256(fh.read()).hexdigest()}  {f}')
with open(f'{OUT}/tn5b_SHA256.txt', 'w', encoding='utf-8', newline='\n') as fh:
    fh.write('\n'.join(dong) + '\n')
print('\n'.join(dong))

import shutil
shutil.make_archive('/content/TN5B_KetQua', 'zip', OUT)
from google.colab import files
files.download('/content/TN5B_KetQua.zip')